<div style="display:block;width:100%;margin:auto;" direction=rtl align=center>
    <br><br>
    <div style="width:100%;margin:100;display:block;background-color:#fff0;" display=block align=center>
        <table style="border-style:hidden;border-collapse:collapse;">
            <tr>
                <td style="border: none!important;">
                    <img width=130 align=right src="https://i.ibb.co/yXKQmtZ/logo1.png" style="margin:0;" />
                </td>
                <td style="text-align:center;border: none!important;">
                    <h1 align=center><font size=5 color="#045F5F"> <b> Large Language Models </b><br><br>Computer Assignment 1</font></h1>
                </td>
                <td style="border: none!important;">
                    <img width=170 align=left src="https://i.ibb.co/wLjqFkw/logo2.png" style="margin:0;" />
                </td>
            </tr>
        </table>
        <h1> Behzad Jannati / Sobhan Abedi - Final Project -
        <h1> 810103098 /810103081 </h1>
        <h1> Prof. Mohammad Javad Dousti & Yadoulah Yaghoubzadeh
    </div>
</div>

## Pruning With Healing For Qwen-2.5 1.5B

>[Pruning With Healing For Qwen-2.5 1.5B](#updateTitle=true&folderId=1wnJrg2FZWyT5daYGocpxY9s0ZRLVwalr&scrollTo=XONqU5vYlqki)

>>[Libraries And Setup](#updateTitle=true&folderId=1wnJrg2FZWyT5daYGocpxY9s0ZRLVwalr&scrollTo=HUHaWokekZjp)

>>[Pruning with Healing Class Implementation](#updateTitle=true&folderId=1wnJrg2FZWyT5daYGocpxY9s0ZRLVwalr&scrollTo=IzzjGwXdkkJz)

>>[Main Execution Function](#updateTitle=true&folderId=1wnJrg2FZWyT5daYGocpxY9s0ZRLVwalr&scrollTo=R1Eh2Ht35KjU)

>>[Analysis and Results Visualization](#updateTitle=true&folderId=1wnJrg2FZWyT5daYGocpxY9s0ZRLVwalr&scrollTo=JwpyhUUOlXXh)

>>[Comparison Between Pruned and Healed Models](#updateTitle=true&folderId=1wnJrg2FZWyT5daYGocpxY9s0ZRLVwalr&scrollTo=c8487_tbDVhS)



In [ ]:
!git config --global credential.helper store

In [ ]:
!hf auth login --token HF_TOKEN --add-to-git-credential

Token is valid (permission: fineGrained).
The token `llm` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `llm`


##Libraries And Setup

In [ ]:
# Install required packages (if not already installed)
!pip install transformers>=4.36.0
!pip install bitsandbytes>=0.41.0
!pip install accelerate>=0.25.0
!pip install datasets>=2.14.0
!pip install sentencepiece==0.1.99
!pip install protobuf==3.20.3
!pip install torch>=2.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 40.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for sentencepiece: filename=sentencepiece-0.1.99-cp312-cp312-linux_x86_64.whl size=1266900 sha256=78dd515a715db3dba237c04750c879e783ec9365e32340abcecc8355415c6a18
  Stored in directory: /root/.cache/pip/wheels/e0/8c/e0/65e33b1f4b8462dfc537a0cac02e5c03e1207564c300e4bde5
Successfully built sentencepiece
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.1
    Uninstalling sentencepiece-0.2.1:
      Successfully uninstalled sentencepiece-0.2.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

In [ ]:
import os
import gc
import json
import re
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    get_scheduler,
    default_data_collator,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

# Utility functions
def _slugify(name: str) -> str:
    """Convert model name to filesystem-friendly slug"""
    slug = name.strip().lower()
    slug = slug.replace("/", "-").replace(" ", "-")
    slug = re.sub(r"[^a-z0-9\-_.]+", "-", slug)
    slug = re.sub(r"-+", "-", slug).strip("-")
    return slug

def _pick_torch_dtype(dtype: str) -> torch.dtype:
    """Map user string to torch dtype with safety checks"""
    d = dtype.lower()
    if d in {"bf16", "bfloat16"}:
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16 if torch.cuda.is_available() else torch.float32
    if d in {"fp16", "float16", "half"}:
        return torch.float16 if torch.cuda.is_available() else torch.float32
    if d in {"fp32", "float32"}:
        return torch.float32
    return torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (
        torch.float16 if torch.cuda.is_available() else torch.float32
    )

def _arch_probe(model):
    """Identify the architecture type and locate layers and embeddings"""
    # Llama/Mistral/Gemma/Qwen style
    if hasattr(model, "model") and hasattr(model.model, "layers") and hasattr(model.model, "embed_tokens"):
        return model.model.layers, model.model.embed_tokens

    # Some Qwen variants expose top-level 'layers' and 'embed_tokens'
    if hasattr(model, "layers") and hasattr(model, "embed_tokens"):
        return model.layers, model.embed_tokens

    # GPT-2 family
    if hasattr(model, "transformer") and hasattr(model.transformer, "h") and hasattr(model.transformer, "wte"):
        return model.transformer.h, model.transformer.wte

    # GPT-NeoX family
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers") and hasattr(model.gpt_neox, "embed_in"):
        return model.gpt_neox.layers, model.gpt_neox.embed_in

    # Fallback: treat as generic decoder-only with config count
    if hasattr(model.config, "num_hidden_layers"):
        # Best-effort: try common attribute paths
        for path in [
            ("model", "layers"),
            ("transformer", "h"),
            ("decoder", "layers"),
        ]:
            obj = model
            ok = True
            for p in path:
                if hasattr(obj, p):
                    obj = getattr(obj, p)
                else:
                    ok = False
                    break
            if ok and isinstance(obj, (list, tuple)):
                # try to discover embedding alongside
                embed = None
                for epath in [
                    ("model", "embed_tokens"),
                    ("transformer", "wte"),
                    ("decoder", "embed_tokens"),
                ]:
                    eobj = model
                    eok = True
                    for ep in epath:
                        if hasattr(eobj, ep):
                            eobj = getattr(eobj, ep)
                        else:
                            eok = False
                            break
                    if eok:
                        embed = eobj
                        break
                return obj, embed

    raise ValueError("Model architecture not recognized (failed to locate layers/embeddings).")

In [ ]:
# @title GPU Availability Check
import torch
import gc

# Check the GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 14.7 GB


## Pruning with Healing Class Implementation

In [ ]:
class LayerPrunerWithHealing:
    """
    Prunes transformer layers based on analysis and then performs QLoRA fine-tuning
    to heal the model's performance degradation.
    """

    def __init__(
        self,
        model_name: str = "Qwen/Qwen2.5-1.5B",
        results_dir: str = "./results/pruning_healing",
        device: str = "cuda",
        use_8bit: bool = False,
        use_4bit: bool = True,  # Default to 4-bit for better memory efficiency
        dtype: str = "bf16",
        trust_remote_code: bool = True,
    ):
        """
        Initialize the layer pruner with healing capabilities.

        Args:
            model_name: HF hub model id
            results_dir: Pruning and healing results path
            device: "cuda" or "cpu"
            use_8bit: load in 8-bit (BitsAndBytes)
            use_4bit: load in 4-bit (BitsAndBytes)
            dtype: one of {"bf16","fp16","fp32"}
            trust_remote_code: pass-through to HF loaders
        """
        self.model_name = model_name
        self.slug = _slugify(model_name)
        self.results_dir = results_dir + "/" + self.slug
        self.trust_remote_code = trust_remote_code

        self.device = device if torch.cuda.is_available() and device.startswith("cuda") else "cpu"
        self.use_8bit = bool(use_8bit)
        self.use_4bit = bool(use_4bit)
        self.torch_dtype = _pick_torch_dtype(dtype)

        self.model = None
        self._tokenizer = None
        self.pruned_model = None
        self.healed_model = None

        # Quantization config
        self.bnb_config = None
        if self.use_4bit:
            self.bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=self.torch_dtype,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
        elif self.use_8bit:
            self.bnb_config = BitsAndBytesConfig(
                load_in_8bit=True,
            )

        # Ensure the result path directory exists
        os.makedirs(self.results_dir, exist_ok=True)

    @property
    def tokenizer(self):
        """Lazy-load tokenizer on first access"""
        if getattr(self, "_tokenizer", None) is None:
            self.load_tokenizer()
        return self._tokenizer

    def load_tokenizer(self):
        """Load and configure tokenizer without loading the full model"""
        print(f"Loading tokenizer for {self.model_name}...")
        try:
            self._tokenizer = AutoTokenizer.from_pretrained(
                self.model_name,
                trust_remote_code=self.trust_remote_code
            )
        except ValueError as e:
            # Handle the Qwen2Tokenizer error by using revision parameter
            if "Qwen2Tokenizer" in str(e):
                print("Attempting to load tokenizer with specific revision...")
                self._tokenizer = AutoTokenizer.from_pretrained(
                    self.model_name,
                    trust_remote_code=self.trust_remote_code,
                    revision="refs/pr/1"  # Try with a specific revision
                )

        if self._tokenizer.pad_token is None:
            self._tokenizer.pad_token = getattr(self._tokenizer, "eos_token", None) or self._tokenizer.unk_token
        self._tokenizer.padding_side = "left"

    def load_model(self):
        """Load tokenizer and model with the requested precision/quantization"""
        print(f"Loading model {self.model_name}...")

        # Clear caches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

        # Tokenizer
        if self._tokenizer is None:
            self.load_tokenizer()
        else:
            print("Tokenizer already loaded.")

        # Model kwargs
        model_kwargs = {
            "trust_remote_code": self.trust_remote_code,
            "device_map": "auto" if self.device.startswith("cuda") else None,
        }
        if self.bnb_config is not None:
            model_kwargs["quantization_config"] = self.bnb_config
        else:
            model_kwargs["torch_dtype"] = self.torch_dtype

        # Use AutoModelForCausalLM for broader compatibility
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            **model_kwargs
        )
        self.model.eval()

        # Probe architecture (layers + embedding)
        layers, _ = _arch_probe(self.model)

        # Count layers for user feedback
        num_layers = len(layers)
        print(f"Model loaded with {num_layers} transformer blocks.")
        if torch.cuda.is_available():
            print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

        return num_layers

    def load_pruning_info(self, pruning_file=None):
        """
        Load pruning_sweep information from a CSV file or use default path.
        Returns a dict with pruning_sweep parameters.
        """
        if pruning_file is None:
            pruning_file = f"{self.slug}_aggregate_weighted_mean.csv"

        print(f"Loading pruning_sweep information from {pruning_file}...")

        try:
            df = pd.read_csv(pruning_file)
            # Get the row with the desired block size (typically 14 for Qwen 2.5 1.5B)
            row = df[df["block_size"] == 14].iloc[0]

            pruning_info = {
                "start_layer": int(row["chosen_start_layer"]),
                "end_layer_inclusive": int(row["chosen_end_layer_inclusive"]),
                "block_size": int(row["block_size"]),
                "method": row["method"]
            }
            print(f"Loaded pruning_sweep information: {pruning_info}")
            return pruning_info

        except (FileNotFoundError, IndexError) as e:
            print(f"Error loading pruning_sweep information: {e}")
            print("Using default pruning_sweep parameters (middle layers)...")

            # If we can't load the file, use default parameters
            if self.model is None:
                num_layers = self.load_model()
            else:
                layers, _ = _arch_probe(self.model)
                num_layers = len(layers)

            # Default: prune ~50% of layers from the middle
            block_size = num_layers // 2
            start_layer = (num_layers - block_size) // 2
            end_layer = start_layer + block_size - 1

            return {
                "start_layer": start_layer,
                "end_layer_inclusive": end_layer,
                "block_size": block_size,
                "method": "default"
            }

    def prune_model(self, start_layer=None, end_layer_inclusive=None, save_model=True):
        """
        Prune the model by removing layers from start_layer to end_layer_inclusive.

        Args:
            start_layer: First layer to remove (inclusive)
            end_layer_inclusive: Last layer to remove (inclusive)
            save_model: Whether to save the pruned model

        Returns:
            The pruned model
        """
        # If no specific layers provided, load from pruning_sweep info
        if start_layer is None or end_layer_inclusive is None:
            pruning_info = self.load_pruning_info()
            start_layer = pruning_info["start_layer"]
            end_layer_inclusive = pruning_info["end_layer_inclusive"]

        if self.model is None:
            self.load_model()

        # Get model architecture
        layers, embed_tokens = _arch_probe(self.model)
        num_layers = len(layers)

        # Validate input
        if start_layer < 0 or end_layer_inclusive >= num_layers or start_layer > end_layer_inclusive:
            raise ValueError(f"Invalid layer range: {start_layer}-{end_layer_inclusive} (model has {num_layers} layers)")

        print(f"Step 2: Pruning layers {start_layer} to {end_layer_inclusive} (inclusive)...")
        print(f"Removing {end_layer_inclusive - start_layer + 1} out of {num_layers} layers")

        # Create a deep copy of the model to avoid modifying the original
        self.pruned_model = copy.deepcopy(self.model)
        pruned_layers, pruned_embed = _arch_probe(self.pruned_model)

        # Identify layers to remove and keep
        layers_to_remove = list(range(start_layer, end_layer_inclusive + 1))

        # Llama/Mistral/Qwen style models
        if hasattr(self.pruned_model, "model") and hasattr(self.pruned_model.model, "layers"):
            # Create a new ModuleList with only the layers we want to keep
            new_layers = nn.ModuleList()
            for i in range(num_layers):
                if i not in layers_to_remove:
                    new_layers.append(pruned_layers[i])

            # Replace the layers in the model
            self.pruned_model.model.layers = new_layers

        # Some Qwen variants
        elif hasattr(self.pruned_model, "layers") and hasattr(self.pruned_model, "embed_tokens"):
            # Create a new ModuleList with only the layers we want to keep
            new_layers = nn.ModuleList()
            for i in range(num_layers):
                if i not in layers_to_remove:
                    new_layers.append(pruned_layers[i])

            # Replace the layers in the model
            self.pruned_model.layers = new_layers

        # GPT-2 family
        elif hasattr(self.pruned_model, "transformer") and hasattr(self.pruned_model.transformer, "h"):
            # Create a new ModuleList with only the layers we want to keep
            new_layers = nn.ModuleList()
            for i in range(num_layers):
                if i not in layers_to_remove:
                    new_layers.append(pruned_layers[i])

            # Replace the layers in the model
            self.pruned_model.transformer.h = new_layers

        # GPT-NeoX family
        elif hasattr(self.pruned_model, "gpt_neox") and hasattr(self.pruned_model.gpt_neox, "layers"):
            # Create a new ModuleList with only the layers we want to keep
            new_layers = nn.ModuleList()
            for i in range(num_layers):
                if i not in layers_to_remove:
                    new_layers.append(pruned_layers[i])

            # Replace the layers in the model
            self.pruned_model.gpt_neox.layers = new_layers

        else:
            raise ValueError("Unsupported model architecture for pruning_sweep")

        # Update model config to reflect the new number of layers
        if hasattr(self.pruned_model.config, "num_hidden_layers"):
            self.pruned_model.config.num_hidden_layers = num_layers - len(layers_to_remove)

        # Save the pruned model if requested
        if save_model:
            output_dir = os.path.join(self.results_dir, f"pruned_{start_layer}_{end_layer_inclusive}")
            os.makedirs(output_dir, exist_ok=True)

            # Save the model and tokenizer
            self.pruned_model.save_pretrained(output_dir)
            self.tokenizer.save_pretrained(output_dir)
            print(f"Pruned model saved to {output_dir}")

        return self.pruned_model

    def prepare_dataset(self, dataset_name="wikitext", dataset_config="wikitext-2-raw-v1", split="train",
                       max_samples=1000, sequence_length=512):
        """
        Prepare a dataset for healing fine-tuning.

        Args:
            dataset_name: Name of the dataset to load
            dataset_config: Configuration of the dataset
            split: Split to use (train, validation, test)
            max_samples: Maximum number of samples to use
            sequence_length: Maximum sequence length

        Returns:
            Prepared dataset ready for training
        """
        print(f"Step 3: Preparing {dataset_name} dataset for healing...")

        # Load dataset
        dataset = load_dataset(dataset_name, name=dataset_config, split=split)

        # Subsample if needed
        if max_samples and max_samples < len(dataset):
            dataset = dataset.select(range(max_samples))

        # Tokenize function
        def tokenize_function(examples):
            texts = examples["text"]
            # Filter out empty strings
            texts = [t for t in texts if t.strip()]

            # Tokenize with truncation and padding
            tokenized = self.tokenizer(
                texts,
                truncation=True,
                max_length=sequence_length,
                return_tensors="pt",
                padding="max_length"
            )

            # Prepare labels (same as input_ids for causal LM)
            tokenized["labels"] = tokenized["input_ids"].clone()
            return tokenized

        # Apply tokenization
        tokenized_dataset = dataset.map(
            tokenize_function,
            batched=True,
            remove_columns=dataset.column_names,
            desc="Tokenizing dataset"
        )

        print(f"Dataset prepared with {len(tokenized_dataset)} samples")
        return tokenized_dataset

    def heal_model(self, dataset=None, output_dir=None, lora_r=16, lora_alpha=32, lora_dropout=0.05,
                  batch_size=4, learning_rate=2e-4, num_epochs=1, warmup_ratio=0.03):
        """
        Heal the pruned model using QLoRA fine-tuning.

        Args:
            dataset: Dataset to use for healing (if None, will prepare wikitext)
            output_dir: Directory to save the healed model
            lora_r: LoRA r dimension
            lora_alpha: LoRA alpha parameter
            lora_dropout: LoRA dropout
            batch_size: Training batch size
            learning_rate: Learning rate
            num_epochs: Number of training epochs
            warmup_ratio: Ratio of warmup steps

        Returns:
            The healed model
        """
        print(f"Step 4: Healing pruned model with QLoRA fine-tuning...")

        if self.pruned_model is None:
            raise ValueError("No pruned model available. Please run prune_model first.")

        # Prepare dataset if not provided
        if dataset is None:
            dataset = self.prepare_dataset()

        # Set up output directory
        if output_dir is None:
            pruning_info = self.load_pruning_info()
            start_layer = pruning_info["start_layer"]
            end_layer = pruning_info["end_layer_inclusive"]
            output_dir = os.path.join(self.results_dir, f"healed_{start_layer}_{end_layer}")
            os.makedirs(output_dir, exist_ok=True)

        # Prepare model for k-bit training
        model = prepare_model_for_kbit_training(self.pruned_model)

        # Configure LoRA
        peft_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        )

        # Get PEFT model
        model = get_peft_model(model, peft_config)
        model.print_trainable_parameters()

        # Set up training arguments
        training_args = {
            "output_dir": output_dir,
            "per_device_train_batch_size": batch_size,
            "gradient_accumulation_steps": 4,
            "learning_rate": learning_rate,
            "num_train_epochs": num_epochs,
            "lr_scheduler_type": "cosine",
            "warmup_ratio": warmup_ratio,
            "logging_steps": 10,
            "save_steps": 100,
            "save_total_limit": 3,
            "fp16": self.torch_dtype == torch.float16,
            "bf16": self.torch_dtype == torch.bfloat16,
        }

        # Create DataLoader
        train_dataloader = DataLoader(
            dataset,
            batch_size=training_args["per_device_train_batch_size"],
            collate_fn=default_data_collator,
            shuffle=True
        )

        # Optimizer
        optimizer = torch.optim.AdamW(model.parameters(), lr=training_args["learning_rate"])

        # Learning rate scheduler
        num_update_steps_per_epoch = len(train_dataloader) // training_args["gradient_accumulation_steps"]
        max_train_steps = num_epochs * num_update_steps_per_epoch
        warmup_steps = int(max_train_steps * warmup_ratio)

        lr_scheduler = get_scheduler(
            name=training_args["lr_scheduler_type"],
            optimizer=optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=max_train_steps,
        )

        # Training loop
        model.train()
        progress_bar = tqdm(range(max_train_steps))
        completed_steps = 0

        for epoch in range(num_epochs):
            for step, batch in enumerate(train_dataloader):
                # Move batch to device
                batch = {k: v.to(self.device) for k, v in batch.items()}

                # Forward pass
                outputs = model(**batch)
                loss = outputs.loss

                # Backward pass
                loss.backward()

                # Update weights
                if (step + 1) % training_args["gradient_accumulation_steps"] == 0:
                    optimizer.step()
                    lr_scheduler.step()
                    optimizer.zero_grad()
                    completed_steps += 1
                    progress_bar.update(1)
                    progress_bar.set_description(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

                    # Save checkpoint
                    if completed_steps % training_args["save_steps"] == 0:
                        model.save_pretrained(os.path.join(output_dir, f"checkpoint-{completed_steps}"))

                if completed_steps >= max_train_steps:
                    break

        # Save final model
        model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        print(f"Healed model saved to {output_dir}")

        self.healed_model = model
        return model

    def evaluate_perplexity(self, model=None, dataset_name="wikitext", dataset_config="wikitext-2-raw-v1",
                          split="validation", max_samples=100):
        """
        Evaluate the perplexity of a model on a dataset.

        Args:
            model: The model to evaluate (if None, uses self.model)
            dataset_name: Name of the dataset to load
            dataset_config: Configuration of the dataset
            split: Split to use (train, validation, test)
            max_samples: Maximum number of samples to evaluate

        Returns:
            Perplexity score
        """
        if model is None:
            if self.healed_model is not None:
                model = self.healed_model
            elif self.pruned_model is not None:
                model = self.pruned_model
            else:
                model = self.model

        if model is None:
            raise ValueError("No model available for evaluation")

        # Load dataset
        dataset = load_dataset(dataset_name, name=dataset_config, split=split)

        # Prepare dataset
        texts = []
        for sample in dataset:
            if len(texts) >= max_samples:
                break
            txt = (sample.get("text") or "").strip()
            if len(txt) > 50:  # Skip empty or very short texts
                texts.append(txt)

        # Tokenize texts
        encodings = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
        input_ids = encodings["input_ids"].to(self.device)
        attention_mask = encodings["attention_mask"].to(self.device)

        # Compute perplexity
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
            loss = outputs.loss

        perplexity = torch.exp(loss).item()
        return perplexity

    def compare_models(self, start_layer=None, end_layer_inclusive=None):
        """
        Compare the original, pruned, and healed models.

        Args:
            start_layer: First layer to remove (inclusive)
            end_layer_inclusive: Last layer to remove (inclusive)

        Returns:
            Dictionary with comparison results
        """
        # If no specific layers provided, load from pruning_sweep info
        if start_layer is None or end_layer_inclusive is None:
            pruning_info = self.load_pruning_info()
            start_layer = pruning_info["start_layer"]
            end_layer_inclusive = pruning_info["end_layer_inclusive"]

        print(f"Step 5: Comparing model performance...")

        # Ensure models are loaded
        if self.model is None:
            self.load_model()

        # Prune the model if not already pruned
        if self.pruned_model is None:
            self.prune_model(start_layer, end_layer_inclusive)

        # Evaluate perplexity
        print("Evaluating original model...")
        original_ppl = self.evaluate_perplexity(self.model)

        print("Evaluating pruned model...")
        pruned_ppl = self.evaluate_perplexity(self.pruned_model)

        healed_ppl = None
        if self.healed_model is not None:
            print("Evaluating healed model...")
            healed_ppl = self.evaluate_perplexity(self.healed_model)

        # Calculate memory usage
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            self.model.to(self.device)
            torch.cuda.synchronize()
            original_mem = torch.cuda.max_memory_allocated() / 1024**3  # GB

            torch.cuda.empty_cache()
            self.pruned_model.to(self.device)
            torch.cuda.synchronize()
            pruned_mem = torch.cuda.max_memory_allocated() / 1024**3  # GB

            healed_mem = None
            if self.healed_model is not None:
                torch.cuda.empty_cache()
                self.healed_model.to(self.device)
                torch.cuda.synchronize()
                healed_mem = torch.cuda.max_memory_allocated() / 1024**3  # GB
        else:
            original_mem = 0
            pruned_mem = 0
            healed_mem = 0

        # Count parameters
        original_params = sum(p.numel() for p in self.model.parameters())
        pruned_params = sum(p.numel() for p in self.pruned_model.parameters())

        healed_params = None
        if self.healed_model is not None:
            healed_params = sum(p.numel() for p in self.healed_model.parameters())

        # Prepare results
        results = {
            "model_name": self.model_name,
            "pruning_sweep": {
                "start_layer": start_layer,
                "end_layer_inclusive": end_layer_inclusive,
                "layers_removed": end_layer_inclusive - start_layer + 1,
            },
            "metrics": {
                "original_perplexity": original_ppl,
                "pruned_perplexity": pruned_ppl,
                "pruned_perplexity_increase": pruned_ppl - original_ppl,
                "pruned_perplexity_increase_percent": (pruned_ppl - original_ppl) / original_ppl * 100,
            },
            "memory": {
                "original_memory_gb": original_mem,
                "pruned_memory_gb": pruned_mem,
                "pruned_memory_reduction_gb": original_mem - pruned_mem,
                "pruned_memory_reduction_percent": (original_mem - pruned_mem) / original_mem * 100 if original_mem > 0 else 0,
            },
            "parameters": {
                "original_params": original_params,
                "pruned_params": pruned_params,
                "pruned_params_reduction": original_params - pruned_params,
                "pruned_params_reduction_percent": (original_params - pruned_params) / original_params * 100,
            }
        }

        # Add healed model metrics if available
        if healed_ppl is not None:
            results["metrics"]["healed_perplexity"] = healed_ppl
            results["metrics"]["healed_perplexity_increase"] = healed_ppl - original_ppl
            results["metrics"]["healed_perplexity_increase_percent"] = (healed_ppl - original_ppl) / original_ppl * 100
            results["metrics"]["healing_improvement"] = pruned_ppl - healed_ppl
            results["metrics"]["healing_improvement_percent"] = (pruned_ppl - healed_ppl) / pruned_ppl * 100

            results["memory"]["healed_memory_gb"] = healed_mem

            results["parameters"]["healed_params"] = healed_params
            results["parameters"]["healed_params_increase"] = healed_params - pruned_params
            results["parameters"]["healed_params_increase_percent"] = (healed_params - pruned_params) / pruned_params * 100

        # Print summary
        print("\n" + "=" * 60)
        print(f"Model Comparison: {self.model_name}")
        print("=" * 60)
        print(f"Pruned layers {start_layer} to {end_layer_inclusive} (inclusive)")
        print(f"Removed {results['pruning_sweep']['layers_removed']} layers")
        print("\nPerplexity:")
        print(f" Original: {results['metrics']['original_perplexity']:.2f}")
        print(f" Pruned: {results['metrics']['pruned_perplexity']:.2f}")
        print(f" Increase: {results['metrics']['pruned_perplexity_increase']:.2f} ({results['metrics']['pruned_perplexity_increase_percent']:.2f}%)")

        if healed_ppl is not None:
            print(f" Healed: {results['metrics']['healed_perplexity']:.2f}")
            print(f" Healing improvement: {results['metrics']['healing_improvement']:.2f} ({results['metrics']['healing_improvement_percent']:.2f}%)")

        print("\nMemory Usage:")
        print(f" Original: {results['memory']['original_memory_gb']:.2f} GB")
        print(f" Pruned: {results['memory']['pruned_memory_gb']:.2f} GB")
        print(f" Reduction: {results['memory']['pruned_memory_reduction_gb']:.2f} GB ({results['memory']['pruned_memory_reduction_percent']:.2f}%)")

        print("\nParameters:")
        print(f" Original: {results['parameters']['original_params']:,}")
        print(f" Pruned: {results['parameters']['pruned_params']:,}")
        print(f" Reduction: {results['parameters']['pruned_params_reduction']:,} ({results['parameters']['pruned_params_reduction_percent']:.2f}%)")

        if healed_params is not None:
            print(f" Healed: {results['parameters']['healed_params']:,}")
            print(f" LoRA parameters added: {results['parameters']['healed_params_increase']:,}")

        print("=" * 60)

        # Save results
        results_path = os.path.join(self.results_dir, f"comparison_{start_layer}_{end_layer_inclusive}.json")
        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"Comparison results saved to {results_path}")

        # Visualize results
        self.visualize_comparison(results)

        return results

    def visualize_comparison(self, results):
        """
        Visualize the comparison results.

        Args:
            results: Comparison results dictionary
        """
        # Create figure with subplots
        fig, axs = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f"Model Pruning and Healing Comparison: {self.model_name}", fontsize=16)

        # Perplexity comparison
        perplexity_data = [
            results["metrics"]["original_perplexity"],
            results["metrics"]["pruned_perplexity"]
        ]
        perplexity_labels = ["Original", "Pruned"]

        if "healed_perplexity" in results["metrics"]:
            perplexity_data.append(results["metrics"]["healed_perplexity"])
            perplexity_labels.append("Healed")

        axs[0, 0].bar(perplexity_labels, perplexity_data, color=['blue', 'red', 'green'][:len(perplexity_data)])
        axs[0, 0].set_title("Perplexity (lower is better)")
        axs[0, 0].set_ylabel("Perplexity")

        for i, v in enumerate(perplexity_data):
            axs[0, 0].text(i, v + 0.1, f"{v:.2f}", ha='center')

        # Memory usage
        memory_data = [
            results["memory"]["original_memory_gb"],
            results["memory"]["pruned_memory_gb"]
        ]
        memory_labels = ["Original", "Pruned"]

        if "healed_memory_gb" in results["memory"]:
            memory_data.append(results["memory"]["healed_memory_gb"])
            memory_labels.append("Healed")

        axs[0, 1].bar(memory_labels, memory_data, color=['blue', 'red', 'green'][:len(memory_data)])
        axs[0, 1].set_title("Memory Usage")
        axs[0, 1].set_ylabel("Memory (GB)")

        for i, v in enumerate(memory_data):
            axs[0, 1].text(i, v + 0.1, f"{v:.2f} GB", ha='center')

        # Parameters
        param_data = [
            results["parameters"]["original_params"] / 1_000_000,
            results["parameters"]["pruned_params"] / 1_000_000
        ]
        param_labels = ["Original", "Pruned"]

        if "healed_params" in results["parameters"]:
            param_data.append(results["parameters"]["healed_params"] / 1_000_000)
            param_labels.append("Healed")

        axs[1, 0].bar(param_labels, param_data, color=['blue', 'red', 'green'][:len(param_data)])
        axs[1, 0].set_title("Model Parameters")
        axs[1, 0].set_ylabel("Parameters (millions)")

        for i, v in enumerate(param_data):
            axs[1, 0].text(i, v + 0.1, f"{v:.2f}M", ha='center')

        # Pruning details
        axs[1, 1].axis('off')
        pruning_text = (
            f"Pruning Details:\n\n"
            f"- Start layer: {results['pruning_sweep']['start_layer']}\n"
            f"- End layer: {results['pruning_sweep']['end_layer_inclusive']}\n"
            f"- Layers removed: {results['pruning_sweep']['layers_removed']}\n\n"
            f"Parameter reduction: {results['parameters']['pruned_params_reduction_percent']:.2f}%\n"
            f"Memory reduction: {results['memory']['pruned_memory_reduction_percent']:.2f}%\n"
        )

        if "healing_improvement_percent" in results["metrics"]:
            pruning_text += f"\nHealing improvement: {results['metrics']['healing_improvement_percent']:.2f}%"

        axs[1, 1].text(0.1, 0.5, pruning_text, fontsize=12, va='center')

        plt.tight_layout()
        plt.subplots_adjust(top=0.9)

        # Save figure
        fig_path = os.path.join(self.results_dir, f"comparison_visualization.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        print(f"Visualization saved to {fig_path}")

        # Show figure
        plt.show()

    def run_full_pipeline(self, pruning_file=None, dataset=None, lora_r=16, lora_alpha=32,
                        batch_size=4, num_epochs=1):
        """
        Run the full pruning_sweep with healing pipeline.

        Args:
            pruning_file: Path to pruning_sweep information file
            dataset: Dataset to use for healing (if None, will prepare wikitext)
            lora_r: LoRA r dimension
            lora_alpha: LoRA alpha parameter
            batch_size: Training batch size
            num_epochs: Number of training epochs

        Returns:
            Dictionary with comparison results
        """
        # Step 1: Load pruning_sweep information
        pruning_info = self.load_pruning_info(pruning_file)
        start_layer = pruning_info["start_layer"]
        end_layer = pruning_info["end_layer_inclusive"]

        # Step 2: Load original model
        print("Step 1: Loading the original model...")
        self.load_model()

        # Step 3: Prune model
        self.prune_model(start_layer, end_layer)

        # Step 4: Prepare dataset for healing if not provided
        if dataset is None:
            dataset = self.prepare_dataset()

        # Step 5: Heal model with QLoRA
        self.heal_model(
            dataset=dataset,
            lora_r=lora_r,
            lora_alpha=lora_alpha,
            batch_size=batch_size,
            num_epochs=num_epochs
        )

        # Step 6: Compare models
        results = self.compare_models(start_layer, end_layer)

        return results

## Main Execution Function

In [ ]:
def main():
    """Main function to run the pruning_sweep with healing pipeline."""

    # Create the pruner
    pruner = LayerPrunerWithHealing(
        model_name="Qwen/Qwen2.5-1.5B",
        results_dir="./results/pruning_healing",
        use_4bit=True,  # Use 4-bit quantization for memory efficiency
        dtype="bf16",
        trust_remote_code=True,
    )

    # Run the full pipeline
    results = pruner.run_full_pipeline(
        pruning_file="qwen-qwen2.5-1.5b_aggregate_weighted_mean.csv",
        lora_r=16,
        lora_alpha=32,
        batch_size=4,
        num_epochs=1
    )

    return results

if __name__ == "__main__":
    main()

Loading pruning information from qwen-qwen2.5-1.5b_aggregate_weighted_mean.csv...
Loaded pruning information: {'start_layer': 7, 'end_layer_inclusive': 20, 'block_size': 14, 'method': 'weighted_mean_z'}
Step 1: Loading the original model...
Loading model Qwen/Qwen2.5-1.5B...
Loading tokenizer for Qwen/Qwen2.5-1.5B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model loaded with 28 transformer blocks.
GPU memory allocated: 1.08 GB
Step 2: Pruning layers 7 to 20 (inclusive)...
Removing 14 out of 28 layers
Pruned model saved to ./results/pruning_healing/qwen-qwen2.5-1.5b/pruned_7_20
Step 3: Preparing wikitext dataset for healing...


Tokenizing dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset prepared with 647 samples
Step 4: Healing pruned model with QLoRA fine-tuning...
Loading pruning information from qwen-qwen2.5-1.5b_aggregate_weighted_mean.csv...
Loaded pruning information: {'start_layer': 7, 'end_layer_inclusive': 20, 'block_size': 14, 'method': 'weighted_mean_z'}
trainable params: 9,232,384 || all params: 897,777,152 || trainable%: 1.0284


  0%|          | 0/40 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Healed model saved to ./results/pruning_healing/qwen-qwen2.5-1.5b/healed_7_20
Step 5: Comparing model performance...
Evaluating original model...


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.21 GiB. GPU 0 has a total capacity of 14.74 GiB of which 296.12 MiB is free. Process 75709 has 14.45 GiB memory in use. Of the allocated memory 13.48 GiB is allocated by PyTorch, and 851.30 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Analysis and Results Visualization

In [ ]:
def analyze_results(results):
    """
    Analyze and summarize the results of the pruning_sweep and healing process.

    Parameters:
    results: Dictionary with results from the pipeline

    Returns:
    Analysis summary
    """
    summary = {
        "layers_pruned_count": len(results.get("layers_pruned", [])),
        "original_perplexity": None,
        "final_pruned_perplexity": None,
        "final_healed_perplexity": None,
        "perplexity_change_after_pruning": None,
        "perplexity_recovery_after_healing": None,
        "perplexity_comparison_to_original": None
    }

    # Extract perplexity metrics if available
    perplexity_history = results.get("perplexity_history", [])
    if perplexity_history:
        # Find original perplexity
        original_entries = [e for e in perplexity_history if e["stage"] == "original"]
        if original_entries:
            summary["original_perplexity"] = original_entries[0]["perplexity"]

        # Find the last pruned and healed perplexities
        pruned_entries = [e for e in perplexity_history if "pruned" in e["stage"]]
        healed_entries = [e for e in perplexity_history if "healed" in e["stage"]]

        if pruned_entries:
            summary["final_pruned_perplexity"] = pruned_entries[-1]["perplexity"]

        if healed_entries:
            summary["final_healed_perplexity"] = healed_entries[-1]["perplexity"]

        # Calculate changes
        if summary["original_perplexity"] is not None and summary["final_pruned_perplexity"] is not None:
            summary["perplexity_change_after_pruning"] = (
                (summary["final_pruned_perplexity"] - summary["original_perplexity"]) /
                summary["original_perplexity"] * 100
            )

        if summary["final_pruned_perplexity"] is not None and summary["final_healed_perplexity"] is not None:
            recovery_amount = summary["final_pruned_perplexity"] - summary["final_healed_perplexity"]
            degradation = summary["final_pruned_perplexity"] - summary["original_perplexity"]
            if degradation > 0:  # Only calculate if there was degradation
                summary["perplexity_recovery_after_healing"] = (recovery_amount / degradation) * 100

        if summary["original_perplexity"] is not None and summary["final_healed_perplexity"] is not None:
            summary["perplexity_comparison_to_original"] = (
                (summary["final_healed_perplexity"] - summary["original_perplexity"]) /
                summary["original_perplexity"] * 100
            )

    # Create a summary figure
    plt.figure(figsize=(10, 6))

    # Bar chart comparing original, pruned, and healed perplexities
    if all(v is not None for v in [summary["original_perplexity"],
                                  summary["final_pruned_perplexity"],
                                  summary["final_healed_perplexity"]])):

        models = ["Original", "After Pruning", "After Healing"]
        perplexities = [
            summary["original_perplexity"],
            summary["final_pruned_perplexity"],
            summary["final_healed_perplexity"]
        ]

        bars = plt.bar(models, perplexities, color=['blue', 'red', 'green'])

        # Add values on top of bars
        for bar, perplexity in zip(bars, perplexities):
            plt.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{perplexity:.2f}",
                ha='center',
                fontsize=12
            )

        plt.title('Final Perplexity Comparison', fontsize=16)
        plt.ylabel('Perplexity (lower is better)', fontsize=14)
        plt.grid(axis='y', alpha=0.3)

        # Save summary figure
        output_path = os.path.join("./results/pruning_healing/qwen-qwen2.5-1.5b", "summary_comparison.png")
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.tight_layout()
        plt.show()

    # Print summary
    print("\n===== PRUNING AND HEALING SUMMARY =====")
    print(f"Number of layers pruned: {summary['layers_pruned_count']}")

    if summary["original_perplexity"] is not None:
        print(f"Original perplexity: {summary['original_perplexity']:.4f}")

    if summary["final_pruned_perplexity"] is not None:
        print(f"Final perplexity after pruning_sweep: {summary['final_pruned_perplexity']:.4f}")

    if summary["perplexity_change_after_pruning"] is not None:
        print(f"Change after pruning_sweep: {summary['perplexity_change_after_pruning']:.2f}%")

    if summary["final_healed_perplexity"] is not None:
        print(f"Final perplexity after healing: {summary['final_healed_perplexity']:.4f}")

    if summary["perplexity_recovery_after_healing"] is not None:
        print(f"Recovery through healing: {summary['perplexity_recovery_after_healing']:.2f}%")

    if summary["perplexity_comparison_to_original"] is not None:
        print(f"Final model comparison to original: {summary['perplexity_comparison_to_original']:.2f}%")

    print("========================================")

    return summary

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 71)

## Comparison Between Pruned and Healed Models

In [ ]:
def test_models_with_prompts(pruner, prompts=None):
    """Test and compare the original, pruned, and healed models with sample prompts."""

    if prompts is None:
        prompts = [
            "Explain the concept of neural networks in simple terms.",
            "Write a Python function to find the Fibonacci sequence.",
            "What are the main differences between renewable and non-renewable energy sources?",
            "Summarize the plot of Shakespeare's Hamlet.",
            "Explain how quantum computing differs from classical computing."
        ]

    # Ensure models are loaded
    if pruner.model is None:
        pruner.load_model()

    if pruner.pruned_model is None:
        pruning_info = pruner.load_pruning_info()
        pruner.prune_model(pruning_info["start_layer"], pruning_info["end_layer_inclusive"])

    # Results dictionary
    results = {
        "original": [],
        "pruned": [],
        "healed": [] if pruner.healed_model is not None else None
    }

    # Generate responses for each model and prompt
    for prompt in tqdm(prompts, desc="Testing prompts"):
        # Original model
        inputs = pruner.tokenizer(prompt, return_tensors="pt").to(pruner.device)
        outputs = pruner.model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True
        )
        original_response = pruner.tokenizer.decode(outputs[0], skip_special_tokens=True)
        results["original"].append(original_response)

        # Pruned model
        inputs = pruner.tokenizer(prompt, return_tensors="pt").to(pruner.device)
        outputs = pruner.pruned_model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True
        )
        pruned_response = pruner.tokenizer.decode(outputs[0], skip_special_tokens=True)
        results["pruned"].append(pruned_response)

        # Healed model (if available)
        if pruner.healed_model is not None:
            inputs = pruner.tokenizer(prompt, return_tensors="pt").to(pruner.device)
            outputs = pruner.healed_model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.7,
                do_sample=True
            )
            healed_response = pruner.tokenizer.decode(outputs[0], skip_special_tokens=True)
            results["healed"].append(healed_response)

    # Print results
    for i, prompt in enumerate(prompts):
        print("\n" + "=" * 80)
        print(f"Prompt {i+1}: {prompt}")
        print("-" * 80)

        print("\nOriginal Model Response:")
        print(results["original"][i])

        print("\nPruned Model Response:")
        print(results["pruned"][i])

        if results["healed"] is not None:
            print("\nHealed Model Response:")
            print(results["healed"][i])

    return results

# Sample prompts to test
test_prompts = [
    "Explain the concept of neural networks in simple terms.",
    "Write a Python function to find the Fibonacci sequence.",
    "What are the main differences between renewable and non-renewable energy sources?",
    "Summarize the plot of Shakespeare's Hamlet.",
    "Explain how quantum computing differs from classical computing."
]

# test_models_with_prompts(pruner, test_prompts)  # Uncomment after running the main pipeline

## Alternative Code for Optimized Progressive Pruning and Training in Google Colab

In [ ]:
# @title Install required packages
# Install required packages (if not already installed)
!pip install transformers>=4.36.0
!pip install bitsandbytes>=0.41.0
!pip install accelerate>=0.25.0
!pip install datasets>=2.14.0
!pip install sentencepiece==0.1.99
!pip install protobuf==3.20.3
!pip install torch>=2.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 86.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for sentencepiece: filename=sentencepiece-0.1.99-cp312-cp312-linux_x86_64.whl size=1266901 sha256=7266e88b1383d41603f1f733f671ad30264bb61e0077d31cb7460596c9a92a07
  Stored in directory: /root/.cache/pip/wheels/e0/8c/e0/65e33b1f4b8462dfc537a0cac02e5c03e1207564c300e4bde5
Successfully built sentencepiece
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.1
    Uninstalling sentencepiece-0.2.1:
      Successfully uninstalled sentencepiece-0.2.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 11.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source o

In [ ]:
# @title Check GPU Availability
import torch
import gc

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 14.7 GB


In [ ]:
# Import necessary libraries
import os
import gc
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional, Tuple, Union, Any
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
import torch.nn as nn

In [ ]:
# Import necessary libraries
import os
import gc
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional, Tuple, Union, Any
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    get_scheduler
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    TaskType,
    prepare_model_for_kbit_training
)
import torch.nn as nn

class PruningWithHealing:
    """
    Class for pruning_sweep transformer layers from LLMs and then healing the model
    through QLoRA fine-tuning to recover performance.
    """

    def __init__(
        self,
        model_name: str = "Qwen/Qwen2.5-1.5B",
        results_dir: str = "./results/pruning_healing",
        device: str = "cuda",
        use_8bit: bool = True, # Default to 8-bit for memory efficiency
        use_4bit: bool = False,
        dtype: str = "bf16",
        pruning_info_path: Optional[str] = None,
        trust_remote_code: bool = True,
    ):
        """
        Initialize the PruningWithHealing class

        Args:
            model_name: HF hub model id
            results_dir: Directory to store results
            device: "cuda" or "cpu"
            use_8bit: Load model in 8-bit precision
            use_4bit: Load model in 4-bit precision
            dtype: One of {"bf16", "fp16", "fp32"}
            pruning_info_path: Path to CSV or JSON pruning_sweep information
            trust_remote_code: Trust remote code for model loading
        """
        self.model_name = model_name
        self.slug = self._slugify(model_name)
        self.results_dir = os.path.join(results_dir, self.slug)
        self.device = device if torch.cuda.is_available() and device.startswith("cuda") else "cpu"
        self.use_8bit = bool(use_8bit)
        self.use_4bit = bool(use_4bit)
        self.torch_dtype = self._pick_torch_dtype(dtype)
        self.trust_remote_code = trust_remote_code

        # Ensure results directory exists
        os.makedirs(self.results_dir, exist_ok=True)

        # Initialize model and tokenizer
        self.model = None
        self._tokenizer = None
        self.pruned_model = None
        self.healed_model = None
        self.pruning_history = []

        # Load pruning_sweep info if provided
        self.pruning_info = None
        if pruning_info_path:
            self.load_pruning_info(pruning_info_path)

        # Quantization config
        self.bnb_config = None
        if self.use_4bit:
            self.bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=self.torch_dtype,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
        elif self.use_8bit:
            bnb_config = BitsAndBytesConfig(
                load_in_8bit=True,
            )

    @property
    def tokenizer(self):
        if getattr(self, "_tokenizer", None) is None:
            self.load_tokenizer()
        return self._tokenizer

    @staticmethod
    def _slugify(name: str) -> str:
        """Create a filesystem-friendly slug from model name."""
        import re
        slug = name.strip().lower()
        slug = slug.replace("/", "-").replace(" ", "-")
        slug = re.sub(r"[^a-z0-9\-_.]+", "-", slug)
        slug = re.sub(r"-+", "-", slug).strip("-")
        return slug

    @staticmethod
    def _pick_torch_dtype(dtype: str) -> torch.dtype:
        """Map user string to torch dtype with safety checks."""
        d = dtype.lower()
        if d in {"bf16", "bfloat16"}:
            if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
                return torch.bfloat16
            return torch.float16 if torch.cuda.is_available() else torch.float32
        if d in {"fp16", "float16", "half"}:
            return torch.float16 if torch.cuda.is_available() else torch.float32
        if d in {"fp32", "float32"}:
            return torch.float32
        # Safe fallback
        return torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (
            torch.float16 if torch.cuda.is_available() else torch.float32
        )

    @staticmethod
    def _arch_probe(model):
        """
        Try to locate:
        - the list of decoder layers
        - the input embedding module

        Supports Llama/Mistral/Gemma/Qwen families, GPT-2/NeoX, and generic HF causal models.
        """
        # Llama/Mistral/Gemma/Qwen style
        if hasattr(model, "model") and hasattr(model.model, "layers") and hasattr(model.model, "embed_tokens"):
            return model.model.layers, model.model.embed_tokens

        # Some Qwen variants expose top-level 'layers' and 'embed_tokens'
        if hasattr(model, "layers") and hasattr(model, "embed_tokens"):
            return model.layers, model.embed_tokens

        # GPT-2 family
        if hasattr(model, "transformer") and hasattr(model.transformer, "h") and hasattr(model.transformer, "wte"):
            return model.transformer.h, model.transformer.wte

        # GPT-NeoX family
        if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers") and hasattr(model.gpt_neox, "embed_in"):
            return model.gpt_neox.layers, model.gpt_neox.embed_in

        # Fallback: treat as generic decoder-only with config count
        if hasattr(model.config, "num_hidden_layers"):
            # Best-effort: try common attribute paths
            for path in [
                ("model", "layers"),
                ("transformer", "h"),
                ("decoder", "layers"),
            ]:
                obj = model
                ok = True
                for p in path:
                    if hasattr(obj, p):
                        obj = getattr(obj, p)
                    else:
                        ok = False
                        break
                if ok and isinstance(obj, (list, tuple)):
                    # try to discover embedding alongside
                    embed = None
                    for epath in [
                        ("model", "embed_tokens"),
                        ("transformer", "wte"),
                        ("decoder", "embed_tokens"),
                    ]:
                        eobj = model
                        eok = True
                        for ep in epath:
                            if hasattr(eobj, ep):
                                eobj = getattr(eobj, ep)
                            else:
                                eok = False
                                break
                        if eok:
                            embed = eobj
                            break
                    return obj, embed

        raise ValueError("Model architecture not recognized (failed to locate layers/embeddings).")

    def load_tokenizer(self):
        """Load/configure tokenizer without loading the full model."""
        print(f"Loading tokenizer for {self.model_name}...")
        self._tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, trust_remote_code=self.trust_remote_code
        )
        if self._tokenizer.pad_token is None:
            self._tokenizer.pad_token = getattr(self._tokenizer, "eos_token", None) or self._tokenizer.unk_token
        self._tokenizer.padding_side = "left"
        print("Tokenizer loaded successfully.")
        return self._tokenizer

    def load_model(self, load_in_8bit=None, load_in_4bit=None):
        """Load model with specified quantization settings."""
        print(f"Loading model {self.model_name}...")

        # Clear GPU memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

        # Override quantization if specified
        use_8bit = self.use_8bit if load_in_8bit is None else load_in_8bit
        use_4bit = self.use_4bit if load_in_4bit is None else load_4bit

        # Ensure tokenizer is loaded
        if self._tokenizer is None:
            self.load_tokenizer()

        # Prepare quantization config
        bnb_config = None
        if use_4bit:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=self.torch_dtype,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
        elif use_8bit:
            bnb_config = BitsAndBytesConfig(
                load_in_8bit=True,
            )

        # Model kwargs
        model_kwargs = {
            "trust_remote_code": self.trust_remote_code,
            "device_map": "auto" if self.device.startswith("cuda") else None,
        }

        if bnb_config is not None:
            model_kwargs["quantization_config"] = bnb_config
        else:
            model_kwargs["torch_dtype"] = self.torch_dtype

        # Load model
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            **model_kwargs
        )
        self.model.eval()

        # Count layers
        layers, _ = self._arch_probe(self.model)
        num_layers = len(layers)
        print(f"Model loaded with {num_layers} transformer blocks.")

        # Report memory usage
        if torch.cuda.is_available():
            print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

        return self.model

    def load_pruning_info(self, path: str):
        """
        Load pruning_sweep information from a CSV or JSON file.

        The file should contain information about which layers to prune.
        """
        print(f"Loading pruning_sweep information from {path}...")

        if path.endswith('.csv'):
            # Assume this is an aggregate analysis CSV
            df = pd.read_csv(path)
            # Get the last row which contains the largest block size to prune
            row = df.iloc[-1]
            self.pruning_info = {
                'start_layer': int(row['chosen_start_layer']),
                'end_layer_inclusive': int(row['chosen_end_layer_inclusive']),
                'block_size': int(row['block_size']),
                'method': row['method'],
            }
        elif path.endswith('.json'):
            # Assume this is a JSON with pruning_sweep recommendations
            with open(path, 'r') as f:
                data = json.load(f)

            # Extract pruning_sweep info from recommendations
            if 'pruning_recommendations_weighted_mean_z' in data:
                # Get recommendation for 50% or highest available
                keys = sorted(data['pruning_recommendations_weighted_mean_z'].keys(),
                             key=lambda x: int(x.strip('%')))
                key = keys[-1] # Use highest pruning_sweep percentage
                rec = data['pruning_recommendations_weighted_mean_z'][key]
                self.pruning_info = {
                    'start_layer': rec['optimal_start_layer'],
                    'end_layer_inclusive': rec['optimal_end_layer'],
                    'block_size': rec['layers_to_remove'],
                    'method': 'weighted_mean_z',
                }
        else:
            raise ValueError(f"Unsupported file format: {path}")

        print(f"Loaded pruning_sweep information: {self.pruning_info}")
        return self.pruning_info

    def prune_model(self, start_layer=None, end_layer_inclusive=None, save_model=True):
        """
        Prune the model by removing specified transformer layers.

        Args:
            start_layer: First layer to remove (inclusive)
            end_layer_inclusive: Last layer to remove (inclusive)
            save_model: Whether to save the pruned model

        Returns:
            The pruned model
        """
        if self.model is None:
            self.load_model()

        # Use provided layers or fall back to pruning_info
        if start_layer is None or end_layer_inclusive is None:
            if self.pruning_info is None:
                raise ValueError("No pruning_sweep information provided. Either load pruning_sweep info or specify layers.")
            start_layer = self.pruning_info['start_layer']
            end_layer_inclusive = self.pruning_info['end_layer_inclusive']

        print(f"Pruning layers {start_layer} to {end_layer_inclusive} (inclusive)...")

        # Get model architecture
        layers, embed_tokens = self._arch_probe(self.model)
        num_layers = len(layers)

        # Validate layer indices
        if start_layer < 0 or end_layer_inclusive >= num_layers:
            raise ValueError(f"Layer indices out of range. Model has {num_layers} layers.")
        if start_layer > end_layer_inclusive:
            raise ValueError(f"Start layer ({start_layer}) must be <= end layer ({end_layer_inclusive}).")

        # Create pruned model
        print("Creating pruned model...")

        # For Qwen models with model.layers structure
        if hasattr(self.model, "model") and hasattr(self.model.model, "layers"):
            # Create a new list of layers excluding pruned ones
            new_layers = []
            for i in range(num_layers):
                if i < start_layer or i > end_layer_inclusive:
                    new_layers.append(layers[i])

            # Replace layers in model
            self.model.model.layers = torch.nn.ModuleList(new_layers)
            self.pruned_model = self.model

            # Update config to reflect new layer count
            if hasattr(self.model.config, "num_hidden_layers"):
                self.model.config.num_hidden_layers = len(new_layers)

        # For models with transformer.h structure (GPT-2 style)
        elif hasattr(self.model, "transformer") and hasattr(self.model.transformer, "h"):
            new_layers = []
            for i in range(num_layers):
                if i < start_layer or i > end_layer_inclusive:
                    new_layers.append(layers[i])

            self.model.transformer.h = torch.nn.ModuleList(new_layers)
            self.pruned_model = self.model

            if hasattr(self.model.config, "n_layer"):
                self.model.config.n_layer = len(new_layers)

        # For direct layers attribute (some Qwen models)
        elif hasattr(self.model, "layers"):
            new_layers = []
            for i in range(num_layers):
                if i < start_layer or i > end_layer_inclusive:
                    new_layers.append(layers[i])

            self.model.layers = torch.nn.ModuleList(new_layers)
            self.pruned_model = self.model

            if hasattr(self.model.config, "num_hidden_layers"):
                self.model.config.num_hidden_layers = len(new_layers)

        else:
            raise ValueError("Unsupported model architecture for pruning_sweep.")

        # Print info about pruned model
        pruned_layers, _ = self._arch_probe(self.pruned_model)
        print(f"Pruned model now has {len(pruned_layers)} transformer blocks.")
        print(f"Removed {num_layers - len(pruned_layers)} layers.")

        # Report memory usage
        if torch.cuda.is_available():
            print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

        # Save pruned model if requested
        if save_model:
            output_dir = os.path.join(self.results_dir, f"pruned_{start_layer}_{end_layer_inclusive}")
            os.makedirs(output_dir, exist_ok=True)

            # Save model and tokenizer
            self.pruned_model.save_pretrained(output_dir)
            self.tokenizer.save_pretrained(output_dir)
            print(f"Pruned model saved to {output_dir}")

        return self.pruned_model

    def prune_single_layer(self, layer_index, save_model=True):
        """
        Prune a single layer from the model.

        Parameters:
        layer_index: Index of the layer to prune
        save_model: Whether to save the pruned model

        Returns:
        The pruned model
        """
        if self.model is None:
            self.load_model()

        print(f"Pruning layer {layer_index}...")

        # Get model architecture
        layers, embed_tokens = self._arch_probe(self.model)
        num_layers = len(layers)

        # Validate layer index
        if layer_index < 0 or layer_index >= num_layers:
            raise ValueError(f"Layer index out of range. Model has {num_layers} layers.")

        # Create a new list of layers excluding the pruned one
        print("Creating pruned model...")

        # For Qwen models with model.layers structure
        if hasattr(self.model, "model") and hasattr(self.model.model, "layers"):
            new_layers = []
            for i in range(num_layers):
                if i != layer_index:
                    new_layers.append(layers[i])

            # Replace layers in the model
            self.model.model.layers = torch.nn.ModuleList(new_layers)
            self.pruned_model = self.model

            # Update config to reflect new number of layers
            if hasattr(self.model.config, "num_hidden_layers"):
                self.model.config.num_hidden_layers = len(new_layers)

        # For models with transformer.h structure (GPT-2 style)
        elif hasattr(self.model, "transformer") and hasattr(self.model.transformer, "h"):
            new_layers = []
            for i in range(num_layers):
                if i != layer_index:
                    new_layers.append(layers[i])

            self.model.transformer.h = torch.nn.ModuleList(new_layers)
            self.pruned_model = self.model

            if hasattr(self.model.config, "n_layer"):
                self.model.config.n_layer = len(new_layers)

        # For direct layers attribute (some Qwen models)
        elif hasattr(self.model, "layers"):
            new_layers = []
            for i in range(num_layers):
                if i != layer_index:
                    new_layers.append(layers[i])

            self.model.layers = torch.nn.ModuleList(new_layers)
            self.pruned_model = self.model

            if hasattr(self.model.config, "num_hidden_layers"):
                self.model.config.num_hidden_layers = len(new_layers)

        else:
            raise ValueError("Unsupported model architecture for pruning_sweep.")


        # Print info about the pruned model
        pruned_layers, _ = self._arch_probe(self.pruned_model)
        print(f"Pruned model now has {len(pruned_layers)} transformer blocks.")
        print(f"Removed 1 layer at index {layer_index}.")

        # Report memory usage
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

        # Save pruned model if requested
        if save_model:
            output_dir = os.path.join(self.results_dir, f"pruned_layer_{layer_index}")
            os.makedirs(output_dir, exist_ok=True)

            # Save model and tokenizer
            self.pruned_model.save_pretrained(output_dir)
            self.tokenizer.save_pretrained(output_dir)
            print(f"Pruned model saved at {output_dir}")

        return self.pruned_model


    def prepare_dataset(self, dataset_name="wikitext", dataset_config="wikitext-2-raw-v1",
                        split="train", max_samples=1000, sequence_length=512, min_text_length=50):
        """
        Prepare a dataset for healing fine-tuning.

        Args:
            dataset_name: Name of dataset to load
            dataset_config: Name of dataset config
            split: Dataset split to use (train, validation, test)
            max_samples: Maximum number of samples to use
            sequence_length: Maximum sequence length for tokenization
            min_text_length: Minimum length of text samples to include
        Returns:
            Prepared dataset
        """
        print(f"Loading dataset {dataset_name}...")
        dataset = load_dataset(dataset_name, name=dataset_config, split=split)

        # Filter out empty or short texts
        dataset = dataset.filter(lambda x: len(x.get("text", "").strip()) >= min_text_length, desc="Filtering short texts")

        # Limit number of samples if specified
        if max_samples and max_samples < len(dataset):
            dataset = dataset.select(range(max_samples))

        # Tokenize dataset
        print(f"Tokenizing dataset with max length {sequence_length}...")

        def tokenize_function(examples):
            # Explicitly handle potential empty texts in the batch
            texts = [text.strip() for text in examples["text"] if text is not None and text.strip()]
            if not texts:
                 # Return empty lists for input_ids and attention_mask if no valid texts
                return {"input_ids": [], "attention_mask": []}

            tokenized = self.tokenizer(
                texts,
                truncation=True,
                max_length=sequence_length,
                padding="max_length",
                return_tensors="pt"
            )
            return tokenized

        tokenized_dataset = dataset.map(
            tokenize_function,
            batched=True,
            remove_columns=dataset.column_names,
            desc="Tokenizing dataset",
        )

        # Filter out samples that result in empty token sequences after tokenization
        tokenized_dataset = tokenized_dataset.filter(
            lambda x: (torch.tensor(x["input_ids"]) != self.tokenizer.pad_token_id).any() if "input_ids" in x and len(x["input_ids"]) > 0 else False,
            desc="Filtering empty token sequences"
        )


        print(f"Dataset prepared with {len(tokenized_dataset)} samples.")
        return tokenized_dataset

    def heal_model(self,
                   dataset=None,
                   output_dir=None,
                   lora_r=8, # Reduced from 16 to save memory
                   lora_alpha=16, # Reduced from 32 to save memory
                   lora_dropout=0.05,
                   learning_rate=5e-5,
                   num_train_epochs=1,
                   per_device_train_batch_size=2, # Reduced batch size for memory efficiency
                   gradient_accumulation_steps=8, # Increased for stability with small batches
                   warmup_ratio=0.03,
                   weight_decay=0.01,
                   save_steps=500,
                   max_steps: Optional[int] = None, # Keep as Optional
                   max_samples=500, # Reduced from 1000 for memory efficiency
                   sequence_length=512):
        """
        Heal the pruned model using QLoRA fine-tuning.

        Args:
            dataset: Dataset to use for fine-tuning (if None, will use wikitext)
            output_dir: Directory to save the healed model
            lora_r: LoRA rank
            lora_alpha: LoRA alpha
            lora_dropout: LoRA dropout
            learning_rate: Learning rate
            num_train_epochs: Number of training epochs
            per_device_train_batch_size: Batch size per device
            gradient_accumulation_steps: Gradient accumulation steps
            warmup_ratio: Ratio of warmup steps
            weight_decay: Weight decay
            save_steps: Steps between checkpoints
            max_steps: Maximum number of training steps (overrides epochs if set)
            max_samples: Maximum number of samples to use
            sequence_length: Maximum sequence length
        Returns:
            The healed model
        """
        if self.pruned_model is None:
            raise ValueError("No pruned model available. Please run prune_model first.")

        # Set up output directory
        if output_dir is None:
            # Use pruning_sweep info for directory naming
            if self.pruning_info is not None:
                start = self.pruning_info['start_layer']
                end = self.pruning_info['end_layer_inclusive']
                output_dir = os.path.join(self.results_dir, f"healed_{start}_{end}")
            else:
                output_dir = os.path.join(self.results_dir, "healed_model")
            os.makedirs(output_dir, exist_ok=True)

        # Prepare dataset if not provided
        if dataset is None:
            dataset = self.prepare_dataset(
                max_samples=max_samples,
                sequence_length=sequence_length
            )

        # Prepare model for k-bit training
        print("Preparing model for QLoRA fine-tuning...")

        # Memory optimization: clear cache before preparing model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

        model = prepare_model_for_kbit_training(self.pruned_model)

        # Configure LoRA - target only key modules to save memory
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"] # Reduced target modules

        peft_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=target_modules,
        )

        # Get PEFT model
        model = get_peft_model(model, peft_config)
        model.print_trainable_parameters()

        # Create data collator
        data_collator = DataCollatorForLanguageModeling(
            tokenizer=self.tokenizer,
            mlm=False,
        )

        # Memory optimization: use gradient checkpointing
        model.gradient_checkpointing_enable()

        # Set up training arguments
        training_args = TrainingArguments(
            output_dir=output_dir,
            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,
            per_device_train_batch_size=per_device_train_batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            warmup_ratio=warmup_ratio,
            weight_decay=0.01, # Keep default weight decay
            save_steps=save_steps,
            logging_steps=50,
            logging_dir=os.path.join(output_dir, "logs"),
            save_total_limit=1, # Keep only 1 checkpoint to save space
            fp16=True if self.torch_dtype == torch.float16 else False,
            bf16=True if self.torch_dtype == torch.bfloat16 else False,
            max_steps=max_steps if max_steps is not None else -1, # Handle None max_steps
            remove_unused_columns=False,
            # Memory optimization
            gradient_checkpointing=True,
            optim="adamw_torch", # Memory-efficient optimizer
            ddp_find_unused_parameters=False,
        )

        # Create trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=dataset,
            data_collator=data_collator,
        )

        # Train model
        print("Starting QLoRA fine-tuning to heal the pruned model...")
        trainer.train()

        # Save final model
        final_dir = os.path.join(output_dir, "final")
        trainer.save_model(final_dir)
        print(f"Healed model saved to {final_dir}")

        self.healed_model = model
        return model

    def evaluate_perplexity(self, model=None, dataset_name="wikitext", dataset_config="wikitext-2-raw-v1",
                           split="validation", max_samples=200, batch_size=8): # Increased batch size
        """
        Evaluate the perplexity of a model on a dataset.

        Args:
            model: Model to evaluate (if None, uses self.model)
            dataset_name: Name of dataset to load
            dataset_config: Name of dataset config
            split: Dataset split to use (train, validation, test)
            max_samples: Maximum number of samples to use
            batch_size: Batch size for evaluation

        Returns:
            Perplexity score or None if no valid tokens were processed
        """
        if model is None:
            model = self.model

        if model is None:
            raise ValueError("No model available for evaluation.")

        # Load dataset
        print(f"Loading dataset {dataset_name} for perplexity evaluation...")
        dataset = load_dataset(dataset_name, name=dataset_config, split=split)

        # Filter out empty or short texts
        dataset = dataset.filter(lambda x: len(x.get("text", "").strip()) > 0, desc="Filtering empty texts") # Ensure non-empty texts

        # Limit number of samples if specified
        if max_samples and max_samples < len(dataset):
            dataset = dataset.select(range(max_samples))

        # Tokenize dataset with padding and truncation
        def tokenize_function(examples):
            # Explicitly handle potential empty texts and ensure batch dimension
            texts = [text.strip() for text in examples["text"] if text is not None and text.strip()]
            if not texts:
                return {"input_ids": [], "attention_mask": []}

            # Ensure truncation and padding for consistent lengths
            tokenized = self.tokenizer(
                texts,
                truncation=True,
                max_length=self.tokenizer.model_max_length, # Use model's max length
                padding="max_length",
                return_tensors="pt"
            )
            return tokenized

        tokenized_dataset = dataset.map(tokenize_function, batched=True)

        # Filter out samples that result in empty token sequences after tokenization
        tokenized_dataset = tokenized_dataset.filter(
             lambda x: (torch.tensor(x["input_ids"]) != self.tokenizer.pad_token_id).any() if "input_ids" in x and len(x["input_ids"]) > 0 else False,
             desc="Filtering empty token sequences after tokenization"
        )


        # Prepare for evaluation
        model.eval()
        nlls = []
        total_length = 0

        # Process dataset in batches
        for i in tqdm(range(0, len(tokenized_dataset), batch_size), desc="Evaluating perplexity"):
            batch = tokenized_dataset[i:i + batch_size]

            # Add check for empty batches after filtering
            if len(batch["input_ids"]) == 0:
                continue

            # Ensure input_ids and attention_mask are Long tensors
            input_ids = torch.stack([torch.tensor(ids, dtype=torch.long) for ids in batch["input_ids"]]).to(self.device)
            attention_mask = torch.stack([torch.tensor(mask, dtype=torch.long) for mask in batch["attention_mask"]]).to(self.device) # Corrected key name
            target_ids = input_ids.clone()

            with torch.no_grad():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=target_ids)
                # Calculate loss only on non-padding tokens
                loss = outputs.loss
                neg_log_likelihood = loss * (attention_mask.sum()) # Sum of non-padding tokens in batch

                nlls.append(neg_log_likelihood.item())
                total_length += attention_mask.sum().item() # Count non-padding tokens


        # Calculate perplexity
        if total_length == 0:
            print("Warning: No valid tokens were processed for perplexity calculation.")
            return None # Return None or a specific value to indicate no calculation was possible

        perplexity = torch.exp(torch.tensor(sum(nlls) / total_length)).item()
        print(f"Perplexity: {perplexity:.4f}")

        return perplexity

    def generate_text(self, model, prompt, max_new_tokens=50, temperature=0.7, top_p=0.9):
        """
        Generate text using the specified model.

        Args:
            model: Model to use for generation
            prompt: Text prompt for generation
            max_new_tokens: Maximum number of new tokens to generate
            temperature: Sampling temperature
            top_p: Top-p sampling parameter

        Returns:
            Generated text
        """
        if model is None:
            raise ValueError("No model provided for text generation.")

        # Tokenize prompt
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        # Memory optimization: clear cache before generation
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

        # Generate text
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                use_cache=True, # Enable KV cache for efficient generation
            )

        # Decode generated text
        generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        return generated_text

    def compare_generations(self, prompt, max_new_tokens=50, temperature=0.7, top_p=0.9):
        """
        Compare text generation between original, pruned, and healed models.

        Args:
            prompt: Text prompt for generation
            max_new_tokens: Maximum number of new tokens to generate
            temperature: Sampling temperature
            top_p: Top-p sampling parameter

        Returns:
            Dictionary with generated text for each model
        """
        results = {}

        # Generate with original model if available
        if self.model is not None:
            print("Generating text with original model...")
            results["original"] = self.generate_text(
                model=self.model,
                prompt=prompt,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p
            )

        # Generate with pruned model if available
        if self.pruned_model is not None:
            print("Generating text with pruned model...")
            results["pruned"] = self.generate_text(
                model=self.pruned_model,
                prompt=prompt,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p
            )

        # Generate with healed model if available
        if self.healed_model is not None:
            print("Generating text with healed model...")
            results["healed"] = self.generate_text(
                model=self.healed_model,
                prompt=prompt,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p
            )

        # Print generated texts
        for model_name, text in results.items():
            print(f"\n--- {model_name.upper()} MODEL GENERATION ---")
            print(text)
            print("-" * 50)

        return results

    def run_progressive_pruning_healing(self,
                                     start_layer=None,
                                     end_layer_inclusive=None,
                                     dataset_name="wikitext",
                                     dataset_config="wikitext-2-raw-v1",
                                     max_samples=500,
                                     sequence_length=512,
                                     lora_r=16,
                                     lora_alpha=32,
                                     num_train_epochs=1,
                                     per_device_train_batch_size=2,
                                     learning_rate=5e-5,
                                     evaluate_each_step=True):
        """
        Run progressive pruning_sweep and healing pipeline, healing after each layer pruning_sweep.

        Parameters:
        start_layer: Starting layer index to prune
        end_layer_inclusive: Ending layer index to prune (inclusive)
        dataset_name: Dataset name for healing
        dataset_config: Dataset config name
        max_samples: Maximum samples for training
        sequence_length: Maximum sequence length
        lora_r: LoRA rank
        lora_alpha: LoRA alpha
        num_train_epochs: Training epochs per healing
        per_device_train_batch_size: Batch size
        learning_rate: Learning rate
        evaluate_each_step: Whether to evaluate after each pruning_sweep+healing step

        Returns:
        Dictionary with results of the pipeline
        """
        results = {
            "perplexity_history": [],
            "layers_pruned": []
        }

        # Use pruning_sweep info if provided layers are None
        if start_layer is None or end_layer_inclusive is None:
            if self.pruning_info is None:
                raise ValueError("No pruning_sweep info provided. Specify start_layer and end_layer_inclusive.")
            start_layer = self.pruning_info['start_layer']
            end_layer_inclusive = self.pruning_info['end_layer_inclusive']

        # 1. Load the original model
        print("Step 1: Loading original model...")
        self.load_model()

        # Evaluate original model
        if evaluate_each_step:
            print("\nEvaluating original model...")
            original_perplexity = self.evaluate_perplexity(
                model=self.model,
                dataset_name=dataset_name,
                dataset_config=dataset_config,
                max_samples=50  # Smaller sample for quicker evaluation
            )
            results["perplexity_history"].append({
                "stage": "original",
                "perplexity": original_perplexity
            })

        # Prepare dataset for healing once
        print("\nPreparing dataset for healing...")
        train_dataset = self.prepare_dataset(
            dataset_name=dataset_name,
            dataset_config=dataset_config,
            split="train",
            max_samples=max_samples,
            sequence_length=sequence_length
        )

        # 2. Progressive pruning_sweep and healing
        print("\nStarting progressive pruning_sweep and healing...")

        # Store the original model for reference
        original_model = self.model

        # Progressively prune and heal layers
        for layer_idx in range(start_layer, end_layer_inclusive + 1):
            print(f"\n--- Pruning and Healing Layer {layer_idx} ---")

            # Prune the current layer
            print(f"Pruning layer {layer_idx}...")
            self.prune_single_layer(layer_idx, save_model=True)
            results["layers_pruned"].append(layer_idx)

            # Evaluate after pruning_sweep if requested
            if evaluate_each_step:
                print(f"Evaluating after pruning_sweep layer {layer_idx}...")
                pruned_perplexity = self.evaluate_perplexity(
                    model=self.pruned_model,
                    dataset_name=dataset_name,
                    dataset_config=dataset_config,
                    max_samples=50
                )
                results["perplexity_history"].append({
                    "stage": f"pruned_layer_{layer_idx}",
                    "perplexity": pruned_perplexity
                })

            # Heal the model
            print(f"Healing after pruning_sweep layer {layer_idx}...")
            output_dir = os.path.join(self.results_dir, f"healed_layer_{layer_idx}")
            self.heal_model(
                dataset=train_dataset,
                output_dir=output_dir,
                lora_r=lora_r,
                lora_alpha=lora_alpha,
                num_train_epochs=num_train_epochs,
                per_device_train_batch_size=per_device_train_batch_size,
                learning_rate=learning_rate,
                max_steps=0  # Use epochs instead of steps
            )

            # Evaluate after healing if requested
            if evaluate_each_step:
                print(f"Evaluating after healing layer {layer_idx}...")
                healed_perplexity = self.evaluate_perplexity(
                    model=self.healed_model,
                    dataset_name=dataset_name,
                    dataset_config=dataset_config,
                    max_samples=50
                )
                results["perplexity_history"].append({
                    "stage": f"healed_layer_{layer_idx}",
                    "perplexity": healed_perplexity
                })

            # Use the healed model as the base for the next pruning_sweep
            self.model = self.healed_model

            # Clear GPU memory
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()

        # 3. Visualize the progressive results
        self.visualize_progressive_results(results["perplexity_history"])

        # 4. Generate text samples with final model
        prompts = [
            "The history of artificial intelligence can be traced to",
            "The benefits of renewable energy include",
            "In the field of quantum computing, researchers have recently discovered that"
        ]

        generation_results = {}
        for i, prompt in enumerate(prompts):
            print(f"\nGenerating with prompt {i+1}: {prompt}")
            # Compare original model with final healed model
            self.model = original_model  # Restore original model
            generation_results[f"prompt_{i+1}"] = {
                "original": self.generate_text(model=original_model, prompt=prompt),
                "final_healed": self.generate_text(model=self.healed_model, prompt=prompt)
            }

        results["generation"] = generation_results

        print("\nProgressive pruning_sweep and healing pipeline completed!")
        return results

    def visualize_progressive_results(self, perplexity_history):
        """
        Visualize the progression of perplexity through pruning_sweep and healing steps.

        Parameters:
        perplexity_history: List of dictionaries with stage and perplexity

        Returns:
        Figure
        """
        # Extract stages and perplexities
        stages = [entry["stage"] for entry in perplexity_history]
        perplexities = [entry["perplexity"] for entry in perplexity_history]

        # Create figure
        plt.figure(figsize=(14, 8))

        # Plot different stages with different colors
        original_idx = [i for i, s in enumerate(stages) if "original" in s]
        pruned_idx = [i for i, s in enumerate(stages) if "pruned" in s]
        healed_idx = [i for i, s in enumerate(stages) if "healed" in s]

        # Plot points
        if original_idx:
            plt.scatter([stages[i] for i in original_idx], [perplexities[i] for i in original_idx],
                       color='blue', label='Original', s=100, zorder=3)
        if pruned_idx:
            plt.scatter([stages[i] for i in pruned_idx], [perplexities[i] for i in pruned_idx],
                       color='red', label='After Pruning', s=80, zorder=3)
        if healed_idx:
            plt.scatter([stages[i] for i in healed_idx], [perplexities[i] for i in healed_idx],
                       color='green', label='After Healing', s=80, zorder=3)

        # Connect points with lines to show progression
        plt.plot(stages, perplexities, 'k--', alpha=0.5, zorder=1)

        # Add labels and title
        plt.title('Perplexity Progression Through Pruning and Healing', fontsize=16)
        plt.ylabel('Perplexity (lower is better)', fontsize=14)
        plt.xlabel('Pipeline Stage', fontsize=14)
        plt.xticks(rotation=45, ha='right', fontsize=10)
        plt.yticks(fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=12)

        # Tight layout to ensure labels fit
        plt.tight_layout()

        # Save the figure
        output_path = os.path.join(self.results_dir, "progressive_perplexity.png")
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Progressive perplexity visualization saved at {output_path}")

        plt.show()

        return plt.gcf()

    def analyze_results(self, results):
        """
        Analyze and summarize the results of the pruning_sweep and healing process.

        Parameters:
        results: Dictionary with results from the pipeline

        Returns:
        Analysis summary
        """
        summary = {
            "layers_pruned_count": len(results.get("layers_pruned", [])),
            "original_perplexity": None,
            "final_pruned_perplexity": None,
            "final_healed_perplexity": None,
            "perplexity_change_after_pruning": None,
            "perplexity_recovery_after_healing": None,
            "perplexity_comparison_to_original": None
        }

        # Extract perplexity metrics if available
        perplexity_history = results.get("perplexity_history", [])
        if perplexity_history:
            # Find original perplexity
            original_entries = [e for e in perplexity_history if e["stage"] == "original"]
            if original_entries:
                summary["original_perplexity"] = original_entries[0]["perplexity"]

            # Find the last pruned and healed perplexities
            pruned_entries = [e for e in perplexity_history if "pruned" in e["stage"]]
            healed_entries = [e for e in perplexity_history if "healed" in e["stage"]]

            if pruned_entries:
                summary["final_pruned_perplexity"] = pruned_entries[-1]["perplexity"]

            if healed_entries:
                summary["final_healed_perplexity"] = healed_entries[-1]["perplexity"]

            # Calculate changes
            if summary["original_perplexity"] is not None and summary["final_pruned_perplexity"] is not None:
                summary["perplexity_change_after_pruning"] = (
                    (summary["final_pruned_perplexity"] - summary["original_perplexity"]) /
                    summary["original_perplexity"] * 100
                )

            if summary["final_pruned_perplexity"] is not None and summary["final_healed_perplexity"] is not None:
                recovery_amount = summary["final_pruned_perplexity"] - summary["final_healed_perplexity"]
                degradation = summary["final_pruned_perplexity"] - summary["original_perplexity"]
                if degradation > 0:  # Only calculate if there was degradation
                    summary["perplexity_recovery_after_healing"] = (recovery_amount / degradation) * 100

            if summary["original_perplexity"] is not None and summary["final_healed_perplexity"] is not None:
                summary["perplexity_comparison_to_original"] = (
                    (summary["final_healed_perplexity"] - summary["original_perplexity"]) /
                    summary["original_perplexity"] * 100
                )

        # Create a summary figure
        plt.figure(figsize=(10, 6))

        # Bar chart comparing original, pruned, and healed perplexities
        if all(v is not None for v in [summary["original_perplexity"],
                                      summary["final_pruned_perplexity"],
                                      summary["final_healed_perplexity"]]):

            models = ["Original", "After Pruning", "After Healing"]
            perplexities = [
                summary["original_perplexity"],
                summary["final_pruned_perplexity"],
                summary["final_healed_perplexity"]
            ]

            bars = plt.bar(models, perplexities, color=['blue', 'red', 'green'])

            # Add values on top of bars
            for bar, perplexity in zip(bars, perplexities):
                plt.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5,
                    f"{perplexity:.2f}",
                    ha='center',
                    fontsize=12
                )

            plt.title('Final Perplexity Comparison', fontsize=16)
            plt.ylabel('Perplexity (lower is better)', fontsize=14)
            plt.grid(axis='y', alpha=0.3)

            # Save summary figure
            output_path = os.path.join(self.results_dir, "summary_comparison.png")
            plt.savefig(output_path, dpi=300, bbox_inches="tight")
            plt.tight_layout()
            plt.show()

        # Print summary
        print("\n===== PRUNING AND HEALING SUMMARY =====")
        print(f"Number of layers pruned: {summary['layers_pruned_count']}")

        if summary["original_perplexity"] is not None:
            print(f"Original perplexity: {summary['original_perplexity']:.4f}")

        if summary["final_pruned_perplexity"] is not None:
            print(f"Final perplexity after pruning_sweep: {summary['final_pruned_perplexity']:.4f}")

        if summary["perplexity_change_after_pruning"] is not None:
            print(f"Change after pruning_sweep: {summary['perplexity_change_after_pruning']:.2f}%")

        if summary["final_healed_perplexity"] is not None:
            print(f"Final perplexity after healing: {summary['final_healed_perplexity']:.4f}")

        if summary["perplexity_recovery_after_healing"] is not None:
            print(f"Recovery through healing: {summary['perplexity_recovery_after_healing']:.2f}%")

        if summary["perplexity_comparison_to_original"] is not None:
            print(f"Final model comparison to original: {summary['perplexity_comparison_to_original']:.2f}%")

        print("========================================")

        return summary

In [ ]:
# @title Run Progressive Pruning and Healing Pipeline
import os
import pandas as pd
import json
import torch
import gc

# Create pruning_sweep info file from existing data
def create_pruning_info():
    pruning_data = [
        {
            "method": "weighted_mean_z",
            "block_size": 14,
            "weight_json": '{"syntax": 0.3333333333333333, "code": 0.3333333333333333, "math": 0.3333333333333333}',
            "chosen_start_layer": 7,
            "chosen_end_layer_inclusive": 20,
            "combined_score": 0.47822816452806655
        }
    ]
    pruning_df = pd.DataFrame(pruning_data)
    pruning_df.to_csv("qwen-qwen2.5-1.5b_aggregate_weighted_mean.csv", index=False)
    print("Created pruning_sweep info file")

def main():
    # Create results directory
    os.makedirs("./results/pruning_healing/qwen-qwen2.5-1.5b", exist_ok=True)

    # Create pruning_sweep info file
    create_pruning_info()

    # Initialize the pruning_sweep with healing pipeline
    pruner = PruningWithHealing(
        model_name="Qwen/Qwen2.5-1.5B",
        results_dir="./results/pruning_healing",
        use_8bit=True,  # Use 8-bit quantization for memory efficiency
        dtype="bf16",
        trust_remote_code=True,  # Important for Qwen models
    )

    # Load pruning_sweep information
    pruning_info = pruner.load_pruning_info("qwen-qwen2.5-1.5b_aggregate_weighted_mean.csv")
    start_layer = pruning_info["start_layer"]
    end_layer = pruning_info["end_layer_inclusive"]

    # Run the progressive pruning_sweep and healing pipeline
    # Instead of pruning_sweep all 14 layers at once, we'll prune them one by one and heal after each
    results = pruner.run_progressive_pruning_healing(
        start_layer=start_layer, # Pass start_layer
        end_layer_inclusive=end_layer, # Pass end_layer_inclusive
        dataset_name="wikitext",
        dataset_config="wikitext-2-raw-v1",
        max_samples=200,  # Reduced for T4 GPU
        sequence_length=512,
        lora_r=16,
        lora_alpha=32,
        num_train_epochs=1,
        per_device_train_batch_size=2,  # Reduced for T4 GPU
        learning_rate=5e-5,
        evaluate_each_step=True  # Evaluate after each pruning_sweep+healing step
    )

    # Analyze the results
    summary = pruner.analyze_results(results)

    # Save results and summary
    with open("./results/pruning_healing/qwen-qwen2.5-1.5b/pipeline_results.json", "w") as f:
        # Convert non-serializable objects to string if necessary
        serializable_results = {k: str(v) if not isinstance(v, (int, float, str, bool, list, dict, type(None))) else v for k, v in results.items()}
        json.dump(serializable_results, f, indent=2)


    with open("./results/pruning_healing/qwen-qwen2.5-1.5b/summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("Pipeline completed and results saved!")

if __name__ == "__main__":
    # Clear GPU memory before starting
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    main()

Created pruning info file
Loading pruning information from qwen-qwen2.5-1.5b_aggregate_weighted_mean.csv...
Loaded pruning information: {'start_layer': 7, 'end_layer_inclusive': 20, 'block_size': 14, 'method': 'weighted_mean_z'}
Step 1: Loading original model...
Loading model Qwen/Qwen2.5-1.5B...
Loading tokenizer for Qwen/Qwen2.5-1.5B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer loaded successfully.
Model loaded with 28 transformer blocks.
GPU memory allocated: 1.67 GB

Evaluating original model...
Loading dataset wikitext for perplexity evaluation...


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Filtering empty token sequences after tokenization:   0%|          | 0/50 [00:00<?, ? examples/s]

Evaluating perplexity:   0%|          | 0/7 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 GiB. GPU 0 has a total capacity of 14.74 GiB of which 9.90 GiB is free. Process 107509 has 4.84 GiB memory in use. Of the allocated memory 4.70 GiB is allocated by PyTorch, and 27.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)